In [ ]:
from google.colab import drive
from pathlib import Path
import json
from datetime import datetime

# Mount persistent storage
drive.mount("/content/drive")

# Project root on Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")

# Create folders without deleting existing files
folders = [
    "data",
    "configs",
    "checkpoints",
    "runs",
    "results",
    "results/metrics",
    "results/figures",
    "results/wsi",
    "logs",
    "src",
    "tests",
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

# Create manifest only if one doesn't already exist
manifest_path = PROJECT_DIR / "project_manifest.json"

if not manifest_path.exists():
    manifest = {
        "project": "Deep Learning-Based Mitosis Detection and WSI Analysis",
        "created_at": datetime.now().astimezone().isoformat(),
        "dataset_repository": "LibreYOLO/mitosis-gjs3g",
        "model": "YOLOv8n",
        "status": "setup"
    }
    manifest_path.write_text(json.dumps(manifest, indent=2))

print("Project workspace:", PROJECT_DIR)
print("\nFolders:")
for folder in folders:
    print(" ", PROJECT_DIR / folder)

print("\nExisting files are preserved.")

In [ ]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU if available.")

In [ ]:
%pip install -q ultralytics huggingface_hub openslide-python opencv-python-headless pyyaml pandas matplotlib seaborn scikit-image

In [ ]:
import ultralytics
import cv2
import pandas as pd
import yaml

print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("Pandas:", pd.__version__)
print("Core imports successful.")

In [ ]:
from huggingface_hub import snapshot_download

DATASET_REPO = "LibreYOLO/mitosis-gjs3g"
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset destination:", DATASET_DIR)

snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    local_dir=str(DATASET_DIR),
)

print("\nDataset downloaded or existing local files reused.")

In [ ]:
from collections import Counter

all_files = [
    p for p in DATASET_DIR.rglob("*")
    if p.is_file()
    and ".cache" not in p.parts
    and ".git" not in p.parts
]

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

image_files = [
    p for p in all_files
    if p.suffix.lower() in image_extensions
]

label_files = [
    p for p in all_files
    if p.suffix.lower() == ".txt"
]

yaml_files = [
    p for p in all_files
    if p.suffix.lower() in {".yaml", ".yml"}
]

print("Total files:", len(all_files))
print("Images:", len(image_files))
print("Text files:", len(label_files))
print("YAML files:", len(yaml_files))

print("\nImage counts by directory:")
for directory, count in Counter(
    str(p.parent.relative_to(DATASET_DIR)) for p in image_files
).most_common():
    print(f"{directory}: {count}")

print("\nYAML files:")
for path in yaml_files:
    print(path.relative_to(DATASET_DIR))

print("\nSample file paths:")
for path in all_files[:40]:
    print(path.relative_to(DATASET_DIR))

In [ ]:
import yaml

for yaml_path in yaml_files:
    print(f"\n{'=' * 60}")
    print("FILE:", yaml_path.relative_to(DATASET_DIR))
    print("=" * 60)

    try:
        config = yaml.safe_load(yaml_path.read_text())
        print(yaml.dump(config, sort_keys=False))
    except Exception as error:
        print("Could not parse this YAML:", repr(error))

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import yaml
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime

# Paths
PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"
AUDIT_DIR = PROJECT_DIR / "results" / "metrics"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATASET_DIR / "data.yaml") as f:
    cfg = yaml.safe_load(f)

root = (DATASET_DIR / cfg.get("path", ".")).resolve()
nc = int(cfg["nc"])
names = cfg.get("names", [])
exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

splits = {
    "train": cfg["train"],
    "val": cfg["val"],
    "test": cfg["test"],
}

def get_label_path(img_path):
    parts = list(img_path.parts)
    if "images" not in parts:
        return None
    parts[parts.index("images")] = "labels"
    return Path(*parts).with_suffix(".txt")

image_rows = []
box_rows = []
errors = []
hash_locations = defaultdict(list)

for split, relative_path in splits.items():
    image_dir = Path(relative_path)
    if not image_dir.is_absolute():
        image_dir = root / image_dir

    image_dir = image_dir.resolve()
    assert image_dir.exists(), f"Missing image directory: {image_dir}"

    files = sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in exts
    )
    print(f"{split}: {len(files)} images")

    for img in files:
        label = get_label_path(img)

        try:
            with Image.open(img) as im:
                width, height = im.size
                im.verify()
        except Exception as e:
            errors.append(f"Unreadable image: {img} ({e})")
            continue

        image_rows.append({
            "split": split,
            "image": str(img.relative_to(DATASET_DIR)),
            "width": width,
            "height": height,
            "label_exists": label is not None and label.exists(),
        })

        # Hash images to detect byte-identical files across splits.
        digest = hashlib.sha256(img.read_bytes()).hexdigest()
        hash_locations[digest].append((split, str(img.relative_to(DATASET_DIR))))

        if label is None or not label.exists():
            errors.append(f"Missing label: {img}")
            continue

        lines = label.read_text().strip().splitlines()

        for line_no, line in enumerate(lines, 1):
            if not line.strip():
                continue

            values = line.split()
            if len(values) != 5:
                errors.append(f"Expected 5 YOLO values: {label}, line {line_no}")
                continue

            try:
                cls_float, xc, yc, bw, bh = map(float, values)
            except ValueError:
                errors.append(f"Non-numeric annotation: {label}, line {line_no}")
                continue

            if not np.isfinite([cls_float, xc, yc, bw, bh]).all():
                errors.append(f"Non-finite annotation: {label}, line {line_no}")
                continue

            if not cls_float.is_integer() or not 0 <= int(cls_float) < nc:
                errors.append(f"Invalid class ID: {label}, line {line_no}")

            if not (0 <= xc <= 1 and 0 <= yc <= 1 and
                    0 < bw <= 1 and 0 < bh <= 1):
                errors.append(f"Invalid normalized box: {label}, line {line_no}")

            class_id = int(cls_float) if cls_float.is_integer() else -1
            class_name = (
                names[class_id]
                if isinstance(names, list) and 0 <= class_id < len(names)
                else str(class_id)
            )

            box_rows.append({
                "split": split,
                "image": str(img.relative_to(DATASET_DIR)),
                "class_id": class_id,
                "class_name": class_name,
                "xc_norm": xc,
                "yc_norm": yc,
                "width_norm": bw,
                "height_norm": bh,
                "box_width_px": bw * width,
                "box_height_px": bh * height,
                "box_area_px": bw * bh * width * height,
                "image_width": width,
                "image_height": height,
            })

images_df = pd.DataFrame(
    image_rows,
    columns=["split", "image", "width", "height", "label_exists"]
)
boxes_df = pd.DataFrame(
    box_rows,
    columns=[
        "split", "image", "class_id", "class_name",
        "xc_norm", "yc_norm", "width_norm", "height_norm",
        "box_width_px", "box_height_px", "box_area_px",
        "image_width", "image_height"
    ]
)

duplicate_rows = []
for digest, locations in hash_locations.items():
    unique_splits = {split for split, _ in locations}
    if len(unique_splits) > 1:
        duplicate_rows.append({
            "sha256": digest,
            "splits": ", ".join(sorted(unique_splits)),
            "files": " | ".join(path for _, path in locations)
        })

duplicates_df = pd.DataFrame(
    duplicate_rows, columns=["sha256", "splits", "files"]
)

# Display results
print("\nIMAGE DIMENSIONS")
if not images_df.empty:
    display(images_df.groupby("split").agg(
        images=("image", "count"),
        width_min=("width", "min"),
        width_max=("width", "max"),
        height_min=("height", "min"),
        height_max=("height", "max"),
        unique_sizes=("image", lambda s: images_df.loc[s.index, ["width", "height"]].drop_duplicates().shape[0]),
        missing_label_files=("label_exists", lambda s: int((~s).sum())),
    ).reset_index())

print("\nANNOTATION SUMMARY")
if not boxes_df.empty:
    display(boxes_df.groupby("split").agg(
        objects=("class_id", "size"),
        median_box_width_px=("box_width_px", "median"),
        median_box_height_px=("box_height_px", "median"),
        min_box_width_px=("box_width_px", "min"),
        min_box_height_px=("box_height_px", "min"),
    ).round(3).reset_index())

    print("\nObjects per class:")
    display(boxes_df.groupby(["split", "class_name"]).size()
            .rename("objects").reset_index())
else:
    print("No valid annotation rows were parsed.")

print("\nEXACT DUPLICATE GROUPS ACROSS SPLITS:", len(duplicates_df))
if not duplicates_df.empty:
    display(duplicates_df.head(20))

print("\nVALIDATION ERRORS:", len(errors))
for err in errors[:50]:
    print("-", err)

# Save reports
images_df.to_csv(AUDIT_DIR / "dataset_images_audit.csv", index=False)
boxes_df.to_csv(AUDIT_DIR / "dataset_annotations_audit.csv", index=False)
duplicates_df.to_csv(AUDIT_DIR / "cross_split_exact_duplicates.csv", index=False)

summary = {
    "split_image_counts": {
        split: int((images_df["split"] == split).sum())
        for split in splits
    },
    "images_audited": len(images_df),
    "annotation_rows_parsed": len(boxes_df),
    "cross_split_exact_duplicate_groups": len(duplicates_df),
    "error_count": len(errors),
    "errors": errors,
    "audit_timestamp": datetime.now().astimezone().isoformat(),
}

(AUDIT_DIR / "dataset_audit_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\nSaved audit reports to:", AUDIT_DIR)

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Rebuild paths independently of earlier notebook variables
PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"
FIGURE_DIR = PROJECT_DIR / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

SPLITS = {
    "Train": DATASET_DIR / "train" / "images",
    "Validation": DATASET_DIR / "valid" / "images",
    "Test": DATASET_DIR / "test" / "images",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def find_label(image_path):
    parts = list(image_path.parts)
    if "images" not in parts:
        raise ValueError(f"Cannot identify images directory in {image_path}")
    parts[parts.index("images")] = "labels"
    return Path(*parts).with_suffix(".txt")

samples = []

for split_name, image_dir in SPLITS.items():
    images = sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not images:
        print(f"WARNING: No images found for {split_name}")
        continue

    selected = random.sample(images, min(4, len(images)))
    samples.extend((split_name, p) for p in selected)

fig, axes = plt.subplots(4, 3, figsize=(15, 19))
axes = axes.ravel()

for ax, (split_name, image_path) in zip(axes, samples):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        ax.set_title(f"{split_name}: unreadable image")
        ax.axis("off")
        continue

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    height, width = image_rgb.shape[:2]
    label_path = find_label(image_path)

    object_count = 0

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                continue

            class_id, xc, yc, bw, bh = map(float, values)

            # YOLO normalized center-format -> pixel corner coordinates
            x1 = int(round((xc - bw / 2) * width))
            y1 = int(round((yc - bh / 2) * height))
            x2 = int(round((xc + bw / 2) * width))
            y2 = int(round((yc + bh / 2) * height))

            x1 = max(0, min(width - 1, x1))
            y1 = max(0, min(height - 1, y1))
            x2 = max(0, min(width - 1, x2))
            y2 = max(0, min(height - 1, y2))

            cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (255, 40, 40), 2)
            object_count += 1

    ax.imshow(image_rgb)
    ax.set_title(
        f"{split_name} | {image_path.name[:22]}\n"
        f"Ground-truth boxes: {object_count}",
        fontsize=9
    )
    ax.axis("off")

for ax in axes[len(samples):]:
    ax.axis("off")

fig.suptitle("Mitosis Dataset: Ground-Truth Annotation Review", fontsize=16)
plt.tight_layout()

figure_path = FIGURE_DIR / "dataset_annotation_quality_check.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("Saved visualization to:", figure_path)

In [ ]:
from pathlib import Path
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"
WSI_DIR = PROJECT_DIR / "results" / "wsi"
WSI_DIR.mkdir(parents=True, exist_ok=True)

TILE_SIZE = 640
random.seed(42)

# Pick four images that have annotations.
image_dir = DATASET_DIR / "train" / "images"
image_paths = sorted(
    p for p in image_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

annotated_images = []
for img_path in image_paths:
    label_path = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
    if label_path.exists() and label_path.read_text().strip():
        annotated_images.append((img_path, label_path))

assert len(annotated_images) >= 4, "Need at least four annotated training images."

chosen = random.sample(annotated_images, 4)

# Build a synthetic 1280 x 1280 mosaic.
mosaic = np.full((2 * TILE_SIZE, 2 * TILE_SIZE, 3), 255, dtype=np.uint8)
expected_boxes = []

tile_origins = [
    (0, 0),
    (TILE_SIZE, 0),
    (0, TILE_SIZE),
    (TILE_SIZE, TILE_SIZE),
]

for (img_path, label_path), (offset_x, offset_y) in zip(chosen, tile_origins):
    image = cv2.imread(str(img_path))
    assert image is not None, f"Could not read {img_path}"
    image = cv2.resize(image, (TILE_SIZE, TILE_SIZE))
    mosaic[offset_y:offset_y + TILE_SIZE, offset_x:offset_x + TILE_SIZE] = image

    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        x1 = (xc - bw / 2) * TILE_SIZE
        y1 = (yc - bh / 2) * TILE_SIZE
        x2 = (xc + bw / 2) * TILE_SIZE
        y2 = (yc + bh / 2) * TILE_SIZE

        expected_boxes.append({
            "class_id": int(cls),
            "x1": x1 + offset_x,
            "y1": y1 + offset_y,
            "x2": x2 + offset_x,
            "y2": y2 + offset_y,
            "source_image": img_path.name,
        })

# Simulate tile-local detections, then map them back into mosaic coordinates.
recovered_boxes = []

for (img_path, label_path), (offset_x, offset_y) in zip(chosen, tile_origins):
    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) != 5:
            continue

        cls, xc, yc, bw, bh = map(float, values)

        tile_box = np.array([
            (xc - bw / 2) * TILE_SIZE,
            (yc - bh / 2) * TILE_SIZE,
            (xc + bw / 2) * TILE_SIZE,
            (yc + bh / 2) * TILE_SIZE,
        ])

        # The actual WSI coordinate transform: add tile origin.
        tile_box[[0, 2]] += offset_x
        tile_box[[1, 3]] += offset_y

        recovered_boxes.append(tile_box)

expected_array = np.array([
    [b["x1"], b["y1"], b["x2"], b["y2"]]
    for b in expected_boxes
])
recovered_array = np.array(recovered_boxes)

assert expected_array.shape == recovered_array.shape
max_abs_error = float(np.max(np.abs(expected_array - recovered_array)))
assert np.allclose(expected_array, recovered_array, atol=1e-6)

print("Synthetic mosaic shape:", mosaic.shape)
print("Tiles:", len(chosen))
print("Ground-truth boxes:", len(expected_boxes))
print("Recovered boxes:", len(recovered_boxes))
print("Maximum coordinate error:", max_abs_error, "pixels")
print("PASS: tile-local to mosaic-coordinate transform is correct.")

mosaic_path = WSI_DIR / "synthetic_mosaic.png"
cv2.imwrite(str(mosaic_path), mosaic)

gt_df = pd.DataFrame(expected_boxes)
gt_path = WSI_DIR / "synthetic_mosaic_ground_truth.csv"
gt_df.to_csv(gt_path, index=False)

# Visualize the mosaic and expected boxes.
display_image = cv2.cvtColor(mosaic, cv2.COLOR_BGR2RGB)
for box in expected_boxes:
    p1 = (int(round(box["x1"])), int(round(box["y1"])))
    p2 = (int(round(box["x2"])), int(round(box["y2"])))
    cv2.rectangle(display_image, p1, p2, (255, 0, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(display_image)
plt.title("Synthetic Mosaic — Known Ground-Truth Boxes")
plt.axis("off")
plt.show()

print("Saved mosaic:", mosaic_path)
print("Saved ground truth:", gt_path)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
WSI_DIR = PROJECT_DIR / "results" / "wsi"
WSI_DIR.mkdir(parents=True, exist_ok=True)

def box_iou(box_a, box_b):
    """IoU for boxes in [x1, y1, x2, y2] format."""
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def global_nms(detections, iou_threshold=0.5):
    """
    Greedy class-aware NMS.
    Each detection: x1, y1, x2, y2, confidence, class_id.
    Coordinates must already be in global slide space.
    """
    if len(detections) == 0:
        return []

    detections = np.asarray(detections, dtype=float)
    kept = []

    for class_id in np.unique(detections[:, 5]):
        class_dets = detections[detections[:, 5] == class_id]
        order = np.argsort(-class_dets[:, 4])

        while len(order):
            best_idx = order[0]
            best = class_dets[best_idx]
            kept.append(best)

            remaining = order[1:]
            order = np.array([
                idx for idx in remaining
                if box_iou(best[:4], class_dets[idx, :4]) <= iou_threshold
            ], dtype=int)

    return sorted(kept, key=lambda d: -d[4])


# Synthetic case:
# A and B are duplicate detections from overlapping tiles.
# C is a separate, nearby mitosis and should survive.
synthetic_detections = [
    [100, 100, 140, 140, 0.91, 0],  # A
    [103, 102, 143, 142, 0.82, 0],  # B: duplicate of A
    [155, 105, 195, 145, 0.88, 0],  # C: separate object
    [300, 300, 340, 340, 0.77, 0],  # D: separate object
]

before = len(synthetic_detections)
after_detections = global_nms(synthetic_detections, iou_threshold=0.5)

print("Detections before NMS:", before)
print("Detections after NMS:", len(after_detections))
print("Kept boxes:")
for det in after_detections:
    print(
        f"box={det[:4]}, confidence={det[4]:.2f}, "
        f"class={int(det[5])}"
    )

# Expected result: A, C and D remain; B is suppressed.
assert len(after_detections) == 3, "NMS did not produce the expected count."
assert any(np.allclose(d[:4], synthetic_detections[0][:4]) for d in after_detections)
assert any(np.allclose(d[:4], synthetic_detections[2][:4]) for d in after_detections)
assert any(np.allclose(d[:4], synthetic_detections[3][:4]) for d in after_detections)

result_df = pd.DataFrame(
    after_detections,
    columns=["x1", "y1", "x2", "y2", "confidence", "class_id"]
)
result_df.to_csv(WSI_DIR / "synthetic_nms_test.csv", index=False)

print("\nPASS: duplicate suppressed; distinct detections retained.")
print("Saved:", WSI_DIR / "synthetic_nms_test.csv")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
SRC_DIR = PROJECT_DIR / "src"
TEST_DIR = PROJECT_DIR / "tests"

SRC_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

wsi_source = r'''
"""Core WSI coordinate and detection utilities."""

import numpy as np


def box_iou(box_a, box_b):
    """Compute IoU for xyxy boxes."""
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


def tile_box_to_slide(box, offset_x, offset_y):
    """Map tile-local xyxy coordinates to global slide coordinates."""
    x1, y1, x2, y2 = map(float, box)
    return [
        x1 + offset_x,
        y1 + offset_y,
        x2 + offset_x,
        y2 + offset_y,
    ]


def global_nms(detections, iou_threshold=0.5):
    """
    Greedy class-aware NMS on global slide coordinates.

    Rows: [x1, y1, x2, y2, confidence, class_id]
    Returns rows sorted by descending confidence.
    """
    if not 0 <= iou_threshold <= 1:
        raise ValueError("iou_threshold must be between 0 and 1")

    if len(detections) == 0:
        return []

    detections = np.asarray(detections, dtype=float)
    if detections.ndim != 2 or detections.shape[1] != 6:
        raise ValueError("Expected detections with shape (N, 6)")

    kept = []

    for class_id in np.unique(detections[:, 5]):
        class_dets = detections[detections[:, 5] == class_id]
        order = np.argsort(-class_dets[:, 4])

        while len(order):
            best_idx = order[0]
            best = class_dets[best_idx]
            kept.append(best)

            remaining = order[1:]
            order = np.array([
                idx for idx in remaining
                if box_iou(best[:4], class_dets[idx, :4]) <= iou_threshold
            ], dtype=int)

    return sorted(kept, key=lambda row: -row[4])
'''

source_path = SRC_DIR / "wsi_inference.py"
source_path.write_text(wsi_source.strip() + "\n")

test_source = r'''
import numpy as np
from wsi_inference import box_iou, tile_box_to_slide, global_nms


def test_tile_box_to_slide():
    assert tile_box_to_slide([10, 20, 30, 40], 640, 1280) == [
        650.0, 1300.0, 670.0, 1320.0
    ]


def test_box_iou_identical_boxes():
    assert box_iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0


def test_global_nms_suppresses_duplicate():
    detections = [
        [100, 100, 140, 140, 0.91, 0],
        [103, 102, 143, 142, 0.82, 0],
        [155, 105, 195, 145, 0.88, 0],
        [300, 300, 340, 340, 0.77, 0],
    ]
    result = global_nms(detections, iou_threshold=0.5)
    assert len(result) == 3
    assert np.isclose(result[0][4], 0.91)


def test_global_nms_keeps_different_classes():
    detections = [
        [100, 100, 140, 140, 0.91, 0],
        [100, 100, 140, 140, 0.85, 1],
    ]
    assert len(global_nms(detections, 0.5)) == 2
'''

test_path = TEST_DIR / "test_wsi_inference.py"
test_path.write_text(test_source.strip() + "\n")

print("Saved:", source_path)
print("Saved:", test_path)

In [ ]:
%pip install -q pytest

In [ ]:
import sys
import pytest

PROJECT_DIR = "/content/drive/MyDrive/mitosis_detection_project"
sys.path.insert(0, f"{PROJECT_DIR}/src")

exit_code = pytest.main([
    "-q",
    f"{PROJECT_DIR}/tests/test_wsi_inference.py",
])

assert exit_code == 0, "Unit tests failed. Fix them before proceeding."
print("All WSI utility tests passed.")

In [ ]:
from pathlib import Path
import yaml
import json
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"
CONFIG_DIR = PROJECT_DIR / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = CONFIG_DIR / "mitosis_data.yaml"

config = {
    "path": str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["Mitosis"],
}

DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))

TRAIN_CONFIG = {
    "model": "yolov8n.pt",
    "imgsz": 640,
    "epochs": 40,
    "batch": 8,
    "patience": 10,
    "seed": 42,
    "deterministic": True,
    "workers": 2,
    "pretrained": True,
    "optimizer": "auto",
    "save": True,
    "save_period": 5,
    "plots": True,
    "validation_set": "valid/images",
    "test_policy": "Reserve final test evaluation until model and confidence threshold are fixed.",
    "created_at": datetime.now().astimezone().isoformat(),
}

CONFIG_PATH = CONFIG_DIR / "baseline_training_config.json"
CONFIG_PATH.write_text(json.dumps(TRAIN_CONFIG, indent=2))

print("Dataset YAML:", DATA_YAML)
print(DATA_YAML.read_text())
print("Training configuration saved:", CONFIG_PATH)

In [ ]:
from pathlib import Path
import torch
import yaml

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"
DATA_YAML = PROJECT_DIR / "configs" / "mitosis_data.yaml"

assert torch.cuda.is_available(), "GPU is not available. Enable a GPU runtime."
assert DATA_YAML.exists(), "Training YAML is missing."

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

root = Path(cfg["path"])
for split in ["train", "val", "test"]:
    image_dir = root / cfg[split]
    assert image_dir.exists(), f"Missing {split} image directory: {image_dir}"
    count = sum(
        p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
        for p in image_dir.rglob("*")
    )
    print(f"{split}: {count} images")
    assert count > 0, f"No images found in {split}"

# Check expected class configuration.
assert cfg["nc"] == 1
assert cfg["names"] == ["Mitosis"]

print("\nGPU:", torch.cuda.get_device_name(0))
print("Input checks passed.")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import json
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA_YAML = PROJECT_DIR / "configs" / "mitosis_data.yaml"
RUNS_DIR = PROJECT_DIR / "runs"
LOGS_DIR = PROJECT_DIR / "logs"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "baseline_yolov8n"

# Fail safely rather than overwrite a previous run.
RUN_DIR = RUNS_DIR / "detect" / RUN_NAME
if RUN_DIR.exists():
    raise FileExistsError(
        f"{RUN_DIR} already exists. Choose a new RUN_NAME so previous results are preserved."
    )

start_time = datetime.now().astimezone()

model = YOLO("yolov8n.pt")  # Downloads pretrained weights if not cached.

results = model.train(
    data=str(DATA_YAML),
    imgsz=640,
    epochs=40,
    batch=8,
    patience=10,
    seed=42,
    deterministic=True,
    workers=2,
    pretrained=True,
    optimizer="auto",
    device=0,
    project=str(RUNS_DIR / "detect"),
    name=RUN_NAME,
    exist_ok=False,
    save=True,
    save_period=5,
    plots=True,
    cache=False,
    verbose=True,
)

end_time = datetime.now().astimezone()

best_path = RUN_DIR / "weights" / "best.pt"
last_path = RUN_DIR / "weights" / "last.pt"

run_record = {
    "run_name": RUN_NAME,
    "started_at": start_time.isoformat(),
    "finished_at": end_time.isoformat(),
    "best_checkpoint_exists": best_path.exists(),
    "last_checkpoint_exists": last_path.exists(),
    "best_checkpoint": str(best_path) if best_path.exists() else None,
    "last_checkpoint": str(last_path) if last_path.exists() else None,
    "run_directory": str(RUN_DIR),
}

record_path = LOGS_DIR / f"{RUN_NAME}_record.json"
record_path.write_text(json.dumps(run_record, indent=2))

print("\nTraining finished.")
print(json.dumps(run_record, indent=2))

assert best_path.exists(), (
    "best.pt was not found in the expected Drive location. Inspect the training output before proceeding."
)
print("\nBest checkpoint persisted to Drive:", best_path)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import json

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
RUN_DIR = PROJECT_DIR / "runs" / "detect" / "baseline_yolov8n"
METRICS_DIR = PROJECT_DIR / "results" / "metrics"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

best_path = RUN_DIR / "weights" / "best.pt"
last_path = RUN_DIR / "weights" / "last.pt"
results_csv = RUN_DIR / "results.csv"

for path in [best_path, last_path, results_csv]:
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {path}")

assert best_path.exists(), "Best checkpoint is missing."
assert results_csv.exists(), "Training results.csv is missing."

history = pd.read_csv(results_csv)
history.columns = history.columns.str.strip()

print("\nAvailable metric columns:")
print(history.columns.tolist())

print("\nLast five epochs:")
display(history.tail().round(4))

# Plot key training curves where available.
candidate_columns = [
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
]

plot_columns = [c for c in candidate_columns if c in history.columns]

if plot_columns:
    ncols = 2
    nrows = (len(plot_columns) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax, col in zip(axes, plot_columns):
        ax.plot(history.index + 1, history[col], marker=".", linewidth=1.5)
        ax.set_title(col)
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)

    for ax in axes[len(plot_columns):]:
        ax.axis("off")

    plt.tight_layout()
    plot_path = FIGURES_DIR / "baseline_training_curves.png"
    fig.savefig(plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved curves:", plot_path)

# Preserve the complete training history as a project artifact.
history_path = METRICS_DIR / "baseline_training_history.csv"
history.to_csv(history_path, index=False)

snapshot = {
    "run_name": "baseline_yolov8n",
    "best_checkpoint": str(best_path),
    "last_checkpoint": str(last_path),
    "training_history": str(history_path),
    "epochs_completed": int(len(history)),
    "best_validation_mAP50": 0.880,
    "best_validation_mAP50_95": 0.504,
    "best_epoch_from_training_log": 19,
    "test_set_evaluated_in_this_run": False,
}

snapshot_path = METRICS_DIR / "baseline_snapshot.json"
snapshot_path.write_text(json.dumps(snapshot, indent=2))

print("\nSaved history:", history_path)
print("Saved snapshot:", snapshot_path)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data" / "mitosis-gjs3g"
BEST_PT = PROJECT_DIR / "runs/detect/baseline_yolov8n/weights/best.pt"
METRICS_DIR = PROJECT_DIR / "results" / "metrics"
FIGURES_DIR = PROJECT_DIR / "results" / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert BEST_PT.exists(), f"Checkpoint missing: {BEST_PT}"

model = YOLO(str(BEST_PT))

VAL_IMAGES = DATASET_DIR / "valid" / "images"
VAL_LABELS = DATASET_DIR / "valid" / "labels"

image_paths = sorted(
    p for p in VAL_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
)

# Predict at a low confidence threshold so we can sweep thresholds later.
LOW_CONF = 0.001
IOU_MATCH = 0.50
CONF_THRESHOLDS = np.round(np.arange(0.05, 0.801, 0.05), 2)

def read_gt(label_path, width, height):
    boxes = []
    if not label_path.exists():
        return np.empty((0, 4), dtype=float)

    for line in label_path.read_text().splitlines():
        vals = line.split()
        if len(vals) != 5:
            continue
        cls, xc, yc, bw, bh = map(float, vals)
        boxes.append([
            (xc - bw / 2) * width,
            (yc - bh / 2) * height,
            (xc + bw / 2) * width,
            (yc + bh / 2) * height,
        ])

    return np.asarray(boxes, dtype=float).reshape(-1, 4)

def iou_xyxy(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def greedy_match(gt_boxes, pred_boxes, pred_scores, iou_threshold=0.5):
    """Confidence-ordered one-to-one matching."""
    order = np.argsort(-pred_scores)
    matched_gt = set()
    tp = 0
    fp = 0

    for pred_idx in order:
        if len(gt_boxes) == 0:
            fp += 1
            continue

        overlaps = np.array([
            iou_xyxy(pred_boxes[pred_idx], gt)
            for gt in gt_boxes
        ])

        # Highest-overlap unmatched GT.
        candidate_order = np.argsort(-overlaps)
        match_idx = next(
            (int(i) for i in candidate_order
             if overlaps[i] >= iou_threshold and int(i) not in matched_gt),
            None
        )

        if match_idx is not None:
            tp += 1
            matched_gt.add(match_idx)
        else:
            fp += 1

    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn

# Cache predictions once, then sweep confidence thresholds.
per_image = []

for idx, image_path in enumerate(image_paths, start=1):
    image = cv2.imread(str(image_path))
    if image is None:
        raise RuntimeError(f"Could not read image: {image_path}")

    h, w = image.shape[:2]
    label_path = VAL_LABELS / f"{image_path.stem}.txt"
    gt_boxes = read_gt(label_path, w, h)

    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=LOW_CONF,
        iou=0.7,          # Ultralytics prediction NMS IoU, not matching IoU
        max_det=300,
        verbose=False,
        device=0,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        pred_boxes = np.empty((0, 4), dtype=float)
        pred_scores = np.empty((0,), dtype=float)
    else:
        pred_boxes = result.boxes.xyxy.cpu().numpy()
        pred_scores = result.boxes.conf.cpu().numpy()

    per_image.append({
        "image": image_path.name,
        "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes,
        "pred_scores": pred_scores,
    })

    if idx % 15 == 0 or idx == len(image_paths):
        print(f"Predicted {idx}/{len(image_paths)} validation images")

rows = []

for threshold in CONF_THRESHOLDS:
    total_tp = total_fp = total_fn = 0

    for item in per_image:
        keep = item["pred_scores"] >= threshold
        preds = item["pred_boxes"][keep]
        scores = item["pred_scores"][keep]

        tp, fp, fn = greedy_match(
            item["gt_boxes"], preds, scores, IOU_MATCH
        )
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    rows.append({
        "confidence_threshold": float(threshold),
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    })

threshold_df = pd.DataFrame(rows)

# Predeclared selection rule: highest F1; choose the higher threshold in a tie.
best_row = threshold_df.sort_values(
    ["F1", "confidence_threshold"],
    ascending=[False, False]
).iloc[0]

chosen_threshold = float(best_row["confidence_threshold"])

print("\nValidation threshold results:")
display(threshold_df.round(4))

print("\nSelected threshold by maximum validation F1:")
display(best_row.to_frame("value"))

threshold_csv = METRICS_DIR / "validation_threshold_analysis.csv"
threshold_df.to_csv(threshold_csv, index=False)

selection_record = {
    "checkpoint": str(BEST_PT),
    "validation_images": len(image_paths),
    "ground_truth_iou_threshold": IOU_MATCH,
    "prediction_nms_iou": 0.7,
    "selection_rule": "Maximum one-to-one greedy-matching F1 on validation; higher confidence wins ties",
    "chosen_confidence_threshold": chosen_threshold,
    "selected_validation_precision": float(best_row["precision"]),
    "selected_validation_recall": float(best_row["recall"]),
    "selected_validation_f1": float(best_row["F1"]),
    "test_evaluated": False,
}

selection_path = METRICS_DIR / "selected_confidence_threshold.json"
selection_path.write_text(__import__("json").dumps(selection_record, indent=2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_df["confidence_threshold"], threshold_df["precision"], marker="o", label="Precision")
ax.plot(threshold_df["confidence_threshold"], threshold_df["recall"], marker="o", label="Recall")
ax.plot(threshold_df["confidence_threshold"], threshold_df["F1"], marker="o", label="F1")
ax.axvline(chosen_threshold, linestyle="--", label=f"Selected = {chosen_threshold:.2f}")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("Validation Confidence-Threshold Analysis (IoU match = 0.50)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()

threshold_plot = FIGURES_DIR / "validation_threshold_analysis.png"
fig.savefig(threshold_plot, dpi=160, bbox_inches="tight")
plt.show()

print("\nSaved table:", threshold_csv)
print("Saved threshold selection:", selection_path)
print("Saved plot:", threshold_plot)
print("FIXED VALIDATION-SELECTED THRESHOLD:", chosen_threshold)

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openslide-tools
%pip install -q openslide-python

In [ ]:
import openslide

print("OpenSlide version:", openslide.__version__)
print("OpenSlide import successful.")

In [ ]:
from pathlib import Path
import urllib.request
import json
import openslide

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
WSI_DIR = PROJECT_DIR / "data" / "wsi_demo"
WSI_DIR.mkdir(parents=True, exist_ok=True)

slide_path = WSI_DIR / "CMU-1-Small-Region.svs"

# Download only if not already present.
if not slide_path.exists():
    url = "https://openslide.cs.cmu.edu/download/openslide-testdata/Aperio/CMU-1-Small-Region.svs"
    urllib.request.urlretrieve(url, slide_path)

slide = openslide.OpenSlide(str(slide_path))

metadata = {
    "path": str(slide_path),
    "dimensions": list(slide.dimensions),
    "level_count": slide.level_count,
    "level_dimensions": [list(d) for d in slide.level_dimensions],
    "level_downsamples": list(slide.level_downsamples),
    "mpp_x": slide.properties.get(openslide.PROPERTY_NAME_MPP_X),
    "mpp_y": slide.properties.get(openslide.PROPERTY_NAME_MPP_Y),
    "vendor": slide.properties.get(openslide.PROPERTY_NAME_VENDOR),
    "objective_power": slide.properties.get(openslide.PROPERTY_NAME_OBJECTIVE_POWER),
}

print(json.dumps(metadata, indent=2))

metadata_path = WSI_DIR / "sample_slide_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))
print("\nSaved metadata:", metadata_path)

slide.close()

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
SRC_DIR = PROJECT_DIR / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)

source = r'''
"""Tile-based whole-slide inference for mitosis detection.

Coordinates in output CSV are level-0 slide pixels.
This module does not claim clinical validity.
"""

from pathlib import Path
import csv
import math

import cv2
import numpy as np
import openslide
from PIL import Image
from ultralytics import YOLO

from wsi_inference import global_nms, tile_box_to_slide


def tissue_mask_from_thumbnail(thumbnail, saturation_threshold=15,
                               brightness_threshold=245):
    """Simple tissue heuristic; may need tuning for different stains/scanners."""
    rgb = np.asarray(thumbnail.convert("RGB"))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    saturation = hsv[:, :, 1]
    brightness = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    return (saturation > saturation_threshold) & (
        brightness < brightness_threshold
    )


def run_wsi_inference(
    slide_path,
    checkpoint_path,
    output_dir,
    confidence=0.35,
    tile_size=640,
    stride=512,
    prediction_iou=0.7,
    global_nms_iou=0.5,
    thumbnail_max_size=2048,
    device=0,
):
    """Run tiled detector inference and export slide-coordinate detections."""
    slide_path = Path(slide_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if not slide_path.exists():
        raise FileNotFoundError(slide_path)
    if not Path(checkpoint_path).exists():
        raise FileNotFoundError(checkpoint_path)
    if tile_size <= 0 or stride <= 0 or stride > tile_size:
        raise ValueError("Require tile_size > 0 and 0 < stride <= tile_size")

    slide = openslide.OpenSlide(str(slide_path))
    try:
        width, height = slide.dimensions
        mpp_x = slide.properties.get(openslide.PROPERTY_NAME_MPP_X)
        mpp_y = slide.properties.get(openslide.PROPERTY_NAME_MPP_Y)

        thumb = slide.get_thumbnail((thumbnail_max_size, thumbnail_max_size))
        mask = tissue_mask_from_thumbnail(thumb)
        mask_image = Image.fromarray((mask * 255).astype(np.uint8))
        mask_path = output_dir / "tissue_mask.png"
        mask_image.save(mask_path)

        # Map level-0 tile centers into thumbnail coordinates for tissue filtering.
        thumb_w, thumb_h = thumb.size
        scale_x = thumb_w / width
        scale_y = thumb_h / height

        model = YOLO(str(checkpoint_path))
        raw_detections = []
        processed_tiles = 0
        skipped_tiles = 0

        for y in range(0, height, stride):
            for x in range(0, width, stride):
                tile_w = min(tile_size, width - x)
                tile_h = min(tile_size, height - y)

                if tile_w <= 0 or tile_h <= 0:
                    continue

                # Skip tiles whose center is background in the thumbnail mask.
                cx = min(thumb_w - 1, max(0, int((x + tile_w / 2) * scale_x)))
                cy = min(thumb_h - 1, max(0, int((y + tile_h / 2) * scale_y)))

                if not mask[cy, cx]:
                    skipped_tiles += 1
                    continue

                region = slide.read_region((x, y), 0, (tile_w, tile_h)).convert("RGB")

                # Pad edge tiles to the detector input tile size.
                tile = Image.new("RGB", (tile_size, tile_size), (255, 255, 255))
                tile.paste(region, (0, 0))

                result = model.predict(
                    source=np.asarray(tile),
                    imgsz=tile_size,
                    conf=confidence,
                    iou=prediction_iou,
                    max_det=300,
                    device=device,
                    verbose=False,
                )[0]

                processed_tiles += 1

                if result.boxes is None or len(result.boxes) == 0:
                    continue

                boxes = result.boxes.xyxy.cpu().numpy()
                scores = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy().astype(int)

                for box, score, cls in zip(boxes, scores, classes):
                    x1, y1, x2, y2 = map(float, box)

                    # Ignore detections entirely inside padded edge regions.
                    x1 = min(max(x1, 0.0), float(tile_w))
                    x2 = min(max(x2, 0.0), float(tile_w))
                    y1 = min(max(y1, 0.0), float(tile_h))
                    y2 = min(max(y2, 0.0), float(tile_h))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    slide_box = tile_box_to_slide(
                        [x1, y1, x2, y2], x, y
                    )

                    raw_detections.append([
                        *slide_box, float(score), int(cls), int(x), int(y)
                    ])

        # Save raw detections before global duplicate suppression.
        raw_csv = output_dir / "detections_raw.csv"
        with raw_csv.open("w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "x1", "y1", "x2", "y2", "confidence",
                "class_id", "tile_x", "tile_y"
            ])
            writer.writerows(raw_detections)

        # Apply global NMS on [x1,y1,x2,y2,confidence,class_id].
        nms_input = [row[:6] for row in raw_detections]
        final_detections = global_nms(
            nms_input, iou_threshold=global_nms_iou
        )

        final_csv = output_dir / "detections.csv"
        with final_csv.open("w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "x1", "y1", "x2", "y2", "confidence", "class_id"
            ])
            writer.writerows(final_detections)

        # Create an overview image and overlay at thumbnail resolution.
        overlay = np.asarray(thumb.convert("RGB")).copy()
        for row in final_detections:
            x1, y1, x2, y2, score, cls = row
            tx1 = int(round(x1 * scale_x))
            ty1 = int(round(y1 * scale_y))
            tx2 = int(round(x2 * scale_x))
            ty2 = int(round(y2 * scale_y))

            cv2.rectangle(overlay, (tx1, ty1), (tx2, ty2), (255, 0, 0), 2)
            cv2.putText(
                overlay, f"{score:.2f}",
                (tx1, max(12, ty1 - 3)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1
            )

        overlay_path = output_dir / "detection_overlay.png"
        Image.fromarray(overlay).save(overlay_path)

        summary = {
            "slide_path": str(slide_path),
            "slide_dimensions_level0": [width, height],
            "level_count": slide.level_count,
            "mpp_x": mpp_x,
            "mpp_y": mpp_y,
            "confidence_threshold": confidence,
            "tile_size": tile_size,
            "stride": stride,
            "prediction_nms_iou": prediction_iou,
            "global_nms_iou": global_nms_iou,
            "tiles_processed": processed_tiles,
            "tiles_skipped": skipped_tiles,
            "raw_detections": len(raw_detections),
            "final_detections": len(final_detections),
            "density_per_mm2": None,
            "density_note": (
                "Not calculated: detector training images have no verified "
                "physical-scale metadata and detections are not ground-truth validated."
            ),
            "interpretation": (
                "Engineering demonstration on a slide without verified mitosis "
                "annotations; detections are unvalidated."
            ),
        }

        import json
        summary_path = output_dir / "summary.json"
        summary_path.write_text(json.dumps(summary, indent=2))

        return summary

    finally:
        slide.close()
'''

module_path = SRC_DIR / "wsi_pipeline.py"
module_path.write_text(source.strip() + "\n")

print("Saved WSI pipeline module:", module_path)

In [ ]:
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from wsi_pipeline import run_wsi_inference

SLIDE_PATH = PROJECT_DIR / "data" / "wsi_demo" / "CMU-1-Small-Region.svs"
CHECKPOINT_PATH = PROJECT_DIR / "runs" / "detect" / "baseline_yolov8n" / "weights" / "best.pt"
OUTPUT_DIR = PROJECT_DIR / "results" / "wsi" / "sample_slide_demo"

summary = run_wsi_inference(
    slide_path=SLIDE_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    output_dir=OUTPUT_DIR,
    confidence=0.35,
    tile_size=640,
    stride=512,
    prediction_iou=0.7,
    global_nms_iou=0.5,
    device=0,
)

print(json.dumps(summary, indent=2))

overlay_path = OUTPUT_DIR / "detection_overlay.png"
overlay = Image.open(overlay_path)

plt.figure(figsize=(10, 10))
plt.imshow(overlay)
plt.title("WSI Engineering Demo — Unvalidated Predictions")
plt.axis("off")
plt.show()

print("\nOutput directory:", OUTPUT_DIR)
print("Raw detections:", OUTPUT_DIR / "detections_raw.csv")
print("Final detections:", OUTPUT_DIR / "detections.csv")
print("Summary:", OUTPUT_DIR / "summary.json")

In [ ]:
from pathlib import Path
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
BEST_PT = PROJECT_DIR / "runs/detect/baseline_yolov8n/weights/best.pt"
DATASET_DIR = PROJECT_DIR / "data/mitosis-gjs3g"
WSI_DIR = PROJECT_DIR / "results/wsi/sample_slide_demo"

CONF = 0.35
model = YOLO(str(BEST_PT))

# A. Validation patch inference
val_images = sorted((DATASET_DIR / "valid/images").glob("*"))
val_images = [p for p in val_images if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

val_counts = []
for p in val_images[:20]:
    result = model.predict(
        source=str(p),
        imgsz=640,
        conf=CONF,
        device=0,
        verbose=False
    )[0]
    n = 0 if result.boxes is None else len(result.boxes)
    val_counts.append({"image": p.name, "detections": n})

val_df = pd.DataFrame(val_counts)
print("Validation patch check (first 20 images):")
display(val_df)
print("Images with at least one detection:", int((val_df["detections"] > 0).sum()), "/", len(val_df))
print("Total detections:", int(val_df["detections"].sum()))

# B. Inspect tissue mask and a set of WSI tiles
slide_path = PROJECT_DIR / "data/wsi_demo/CMU-1-Small-Region.svs"
import openslide

slide = openslide.OpenSlide(str(slide_path))
try:
    width, height = slide.dimensions
    thumbnail = slide.get_thumbnail((1024, 1024)).convert("RGB")
    mask_path = WSI_DIR / "tissue_mask.png"
    mask = np.asarray(Image.open(mask_path).convert("L")) > 0

    print("\nSlide dimensions:", (width, height))
    print("Thumbnail dimensions:", thumbnail.size)
    print("Mask dimensions:", mask.shape)
    print("Tissue-mask fraction:", round(float(mask.mean()), 4))

    # Inspect model predictions on representative full-sized tiles, regardless of mask.
    test_tiles = []
    for x, y in [(0, 0), (512, 512), (1024, 512), (512, 1024)]:
        tile_w = min(640, width - x)
        tile_h = min(640, height - y)
        if tile_w <= 0 or tile_h <= 0:
            continue
        region = slide.read_region((x, y), 0, (tile_w, tile_h)).convert("RGB")
        padded = Image.new("RGB", (640, 640), (255, 255, 255))
        padded.paste(region, (0, 0))
        result = model.predict(
            source=np.asarray(padded),
            imgsz=640,
            conf=0.001,
            device=0,
            verbose=False
        )[0]
        n_low = 0 if result.boxes is None else len(result.boxes)
        if result.boxes is not None and len(result.boxes):
            scores = result.boxes.conf.cpu().numpy()
            n_at_conf = int((scores >= CONF).sum())
            max_score = float(scores.max())
        else:
            n_at_conf = 0
            max_score = 0.0

        test_tiles.append({
            "x": x, "y": y,
            "tile_width": tile_w,
            "tile_height": tile_h,
            "detections_at_0.001": n_low,
            "detections_at_0.35": n_at_conf,
            "max_confidence": max_score,
        })

    display(pd.DataFrame(test_tiles))
finally:
    slide.close()

In [ ]:
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openslide
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
SLIDE_PATH = PROJECT_DIR / "data/wsi_demo/CMU-1-Small-Region.svs"
DIAG_DIR = PROJECT_DIR / "results/wsi/diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

TILE_SIZE = 640
STRIDE = 512
THUMBNAIL_MAX_SIZE = 2048

assert SLIDE_PATH.exists(), f"Slide not found: {SLIDE_PATH}"

slide = openslide.OpenSlide(str(SLIDE_PATH))

try:
    slide_width, slide_height = slide.dimensions

    # Recreate the thumbnail and mask in this cell so their dimensions match.
    thumbnail = slide.get_thumbnail(
        (THUMBNAIL_MAX_SIZE, THUMBNAIL_MAX_SIZE)
    ).convert("RGB")

    thumb_rgb = np.asarray(thumbnail)
    thumb_h, thumb_w = thumb_rgb.shape[:2]

    hsv = cv2.cvtColor(thumb_rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(thumb_rgb, cv2.COLOR_RGB2GRAY)

    tissue_mask = (
        (hsv[:, :, 1] > 15) &
        (gray < 245)
    )

    # Save the exact thumbnail and mask used for this diagnostic.
    thumbnail_path = DIAG_DIR / "diagnostic_thumbnail.png"
    mask_path = DIAG_DIR / "diagnostic_tissue_mask.png"
    thumbnail.save(thumbnail_path)
    Image.fromarray((tissue_mask * 255).astype(np.uint8)).save(mask_path)

    # Build an overlay with mask boundaries visible.
    overlay = thumb_rgb.copy()
    contours, _ = cv2.findContours(
        tissue_mask.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    cv2.drawContours(overlay, contours, -1, (0, 255, 0), 1)

    # Use the exact same center-sampling policy as the pipeline.
    scale_x = thumb_w / slide_width
    scale_y = thumb_h / slide_height

    tile_rows = []

    for y in range(0, slide_height, STRIDE):
        for x in range(0, slide_width, STRIDE):
            tile_w = min(TILE_SIZE, slide_width - x)
            tile_h = min(TILE_SIZE, slide_height - y)

            if tile_w <= 0 or tile_h <= 0:
                continue

            center_x = min(
                thumb_w - 1,
                max(0, int((x + tile_w / 2) * scale_x))
            )
            center_y = min(
                thumb_h - 1,
                max(0, int((y + tile_h / 2) * scale_y))
            )

            center_is_tissue = bool(tissue_mask[center_y, center_x])

            # Fraction of mask pixels in the thumbnail region corresponding
            # to this tile; this is diagnostic only, not the pipeline's filter.
            tx1 = max(0, min(thumb_w, int(x * scale_x)))
            ty1 = max(0, min(thumb_h, int(y * scale_y)))
            tx2 = max(tx1 + 1, min(thumb_w, int((x + tile_w) * scale_x)))
            ty2 = max(ty1 + 1, min(thumb_h, int((y + tile_h) * scale_y)))

            region_mask = tissue_mask[ty1:ty2, tx1:tx2]
            tissue_fraction = (
                float(region_mask.mean()) if region_mask.size else 0.0
            )

            tile_rows.append({
                "x": x,
                "y": y,
                "tile_width": tile_w,
                "tile_height": tile_h,
                "center_x_thumbnail": center_x,
                "center_y_thumbnail": center_y,
                "center_is_tissue": center_is_tissue,
                "thumbnail_region_tissue_fraction": tissue_fraction,
                "decision": "process" if center_is_tissue else "skip",
            })

    tiles_df = pd.DataFrame(tile_rows)
    tiles_df.to_csv(DIAG_DIR / "tile_mask_audit.csv", index=False)

    summary = {
        "slide_dimensions": [slide_width, slide_height],
        "thumbnail_dimensions": [thumb_w, thumb_h],
        "mask_dimensions": list(tissue_mask.shape[::-1]),
        "tissue_mask_fraction": float(tissue_mask.mean()),
        "total_tiles": int(len(tiles_df)),
        "tiles_center_pass": int(tiles_df["center_is_tissue"].sum()),
        "tiles_center_skip": int((~tiles_df["center_is_tissue"]).sum()),
        "tile_size": TILE_SIZE,
        "stride": STRIDE,
        "note": (
            "Center-based tissue filtering can skip tiles containing tissue "
            "if their centers lie in whitespace. Region fraction is diagnostic."
        ),
    }

    (DIAG_DIR / "tile_mask_audit_summary.json").write_text(
        json.dumps(summary, indent=2)
    )

    print(json.dumps(summary, indent=2))
    display(tiles_df)

    fig, axes = plt.subplots(1, 3, figsize=(16, 7))

    axes[0].imshow(thumb_rgb)
    axes[0].set_title("WSI thumbnail")
    axes[0].axis("off")

    axes[1].imshow(tissue_mask, cmap="gray")
    axes[1].set_title("Tissue mask")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title("Mask contours on thumbnail")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    print("Saved diagnostics to:", DIAG_DIR)

finally:
    slide.close()

In [ ]:
from pathlib import Path
import random

import cv2
import numpy as np
import matplotlib.pyplot as plt
import openslide
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATASET_DIR = PROJECT_DIR / "data/mitosis-gjs3g"
SLIDE_PATH = PROJECT_DIR / "data/wsi_demo/CMU-1-Small-Region.svs"
DIAG_DIR = PROJECT_DIR / "results/wsi/diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

train_dir = DATASET_DIR / "train" / "images"
train_paths = sorted(
    p for p in train_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
assert train_paths, "No training images found."

sample_train = random.sample(train_paths, min(4, len(train_paths)))

slide = openslide.OpenSlide(str(SLIDE_PATH))
try:
    width, height = slide.dimensions

    # Choose WSI tile positions that cover different areas.
    positions = [
        (0, 0),
        (min(512, width - 1), min(512, height - 1)),
        (min(1024, width - 1), min(512, height - 1)),
        (min(512, width - 1), min(1024, height - 1)),
    ]

    samples = []

    for path in sample_train:
        img = Image.open(path).convert("RGB").resize((640, 640))
        samples.append(("Training patch", path.name, np.asarray(img)))

    for x, y in positions:
        tw = min(640, width - x)
        th = min(640, height - y)
        region = slide.read_region((x, y), 0, (tw, th)).convert("RGB")

        padded = Image.new("RGB", (640, 640), (255, 255, 255))
        padded.paste(region, (0, 0))

        samples.append((
            f"WSI tile ({x},{y})",
            f"x={x}, y={y}",
            np.asarray(padded)
        ))

    rows = []
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    for ax, (kind, name, rgb) in zip(axes.flat, samples):
        ax.imshow(rgb)
        ax.set_title(f"{kind}\n{name[:24]}", fontsize=8)
        ax.axis("off")

        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        rows.append({
            "source": kind,
            "name": name,
            "width": rgb.shape[1],
            "height": rgb.shape[0],
            "mean_red": float(rgb[:, :, 0].mean()),
            "mean_green": float(rgb[:, :, 1].mean()),
            "mean_blue": float(rgb[:, :, 2].mean()),
            "mean_saturation": float(hsv[:, :, 1].mean()),
            "mean_brightness": float(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).mean()),
        })

    plt.tight_layout()
    comparison_path = DIAG_DIR / "training_vs_wsi_tiles.png"
    fig.savefig(comparison_path, dpi=160, bbox_inches="tight")
    plt.show()

    stats_df = pd.DataFrame(rows)
    display(stats_df.round(2))

    stats_path = DIAG_DIR / "training_vs_wsi_color_statistics.csv"
    stats_df.to_csv(stats_path, index=False)

    print("Saved comparison:", comparison_path)
    print("Saved color statistics:", stats_path)

finally:
    slide.close()

In [ ]:
from pathlib import Path
import csv
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import openslide
from ultralytics import YOLO

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
SLIDE_PATH = PROJECT_DIR / "data/wsi_demo/CMU-1-Small-Region.svs"
CHECKPOINT = PROJECT_DIR / "runs/detect/baseline_yolov8n/weights/best.pt"
OUTPUT_DIR = PROJECT_DIR / "results/wsi/no_mask_diagnostic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIDENCE = 0.35
TILE_SIZE = 640
STRIDE = 512
PREDICTION_IOU = 0.7

assert SLIDE_PATH.exists(), f"Missing slide: {SLIDE_PATH}"
assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"

model = YOLO(str(CHECKPOINT))
slide = openslide.OpenSlide(str(SLIDE_PATH))

tile_rows = []
all_detections = []
start = time.perf_counter()

try:
    width, height = slide.dimensions

    for y in range(0, height, STRIDE):
        for x in range(0, width, STRIDE):
            tile_w = min(TILE_SIZE, width - x)
            tile_h = min(TILE_SIZE, height - y)

            if tile_w <= 0 or tile_h <= 0:
                continue

            region = slide.read_region(
                (x, y), 0, (tile_w, tile_h)
            ).convert("RGB")

            # Match the earlier pipeline's white padding at slide edges.
            tile = Image.new(
                "RGB", (TILE_SIZE, TILE_SIZE), (255, 255, 255)
            )
            tile.paste(region, (0, 0))

            result = model.predict(
                source=np.asarray(tile),
                imgsz=TILE_SIZE,
                conf=CONFIDENCE,
                iou=PREDICTION_IOU,
                max_det=300,
                device=0,
                verbose=False,
            )[0]

            boxes = (
                result.boxes.xyxy.cpu().numpy()
                if result.boxes is not None and len(result.boxes)
                else np.empty((0, 4))
            )
            scores = (
                result.boxes.conf.cpu().numpy()
                if result.boxes is not None and len(result.boxes)
                else np.empty((0,))
            )
            classes = (
                result.boxes.cls.cpu().numpy().astype(int)
                if result.boxes is not None and len(result.boxes)
                else np.empty((0,), dtype=int)
            )

            # Clip predictions to the real tile, excluding white padding.
            tile_count = 0
            for box, score, cls in zip(boxes, scores, classes):
                x1, y1, x2, y2 = map(float, box)
                x1, x2 = np.clip([x1, x2], 0, tile_w)
                y1, y2 = np.clip([y1, y2], 0, tile_h)

                if x2 <= x1 or y2 <= y1:
                    continue

                all_detections.append([
                    x + x1, y + y1, x + x2, y + y2,
                    float(score), int(cls), x, y
                ])
                tile_count += 1

            tile_rows.append({
                "tile_x": x,
                "tile_y": y,
                "detections_at_0.35": tile_count,
                "max_confidence": (
                    float(scores.max()) if len(scores) else 0.0
                ),
            })

    elapsed = time.perf_counter() - start

    tile_df = pd.DataFrame(tile_rows)
    tile_df.to_csv(OUTPUT_DIR / "per_tile_results.csv", index=False)

    raw_df = pd.DataFrame(
        all_detections,
        columns=[
            "x1", "y1", "x2", "y2",
            "confidence", "class_id", "tile_x", "tile_y"
        ],
    )
    raw_df.to_csv(OUTPUT_DIR / "detections_before_global_nms.csv", index=False)

    # Use the already-tested project NMS helper if available.
    import sys
    sys.path.insert(0, str(PROJECT_DIR / "src"))
    from wsi_inference import global_nms

    nms_input = raw_df[
        ["x1", "y1", "x2", "y2", "confidence", "class_id"]
    ].values.tolist()

    final_detections = global_nms(
        nms_input, iou_threshold=0.5
    )

    final_df = pd.DataFrame(
        final_detections,
        columns=["x1", "y1", "x2", "y2", "confidence", "class_id"],
    )
    final_df.to_csv(OUTPUT_DIR / "detections_after_global_nms.csv", index=False)

    # Render detections on a thumbnail.
    thumbnail = slide.get_thumbnail((1532, 2048)).convert("RGB")
    overlay = np.asarray(thumbnail).copy()
    sx = overlay.shape[1] / width
    sy = overlay.shape[0] / height

    import cv2
    for row in final_detections:
        x1, y1, x2, y2, score, cls = row
        p1 = (int(x1 * sx), int(y1 * sy))
        p2 = (int(x2 * sx), int(y2 * sy))
        cv2.rectangle(overlay, p1, p2, (255, 0, 0), 2)
        cv2.putText(
            overlay, f"{score:.2f}",
            (p1[0], max(12, p1[1] - 3)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1
        )

    overlay_path = OUTPUT_DIR / "all_tiles_overlay.png"
    Image.fromarray(overlay).save(overlay_path)

    summary = {
        "slide_dimensions": [width, height],
        "tiles_processed": len(tile_rows),
        "tiles_with_detections": int((tile_df["detections_at_0.35"] > 0).sum()),
        "detections_before_global_nms": len(raw_df),
        "detections_after_global_nms": len(final_df),
        "confidence_threshold": CONFIDENCE,
        "tile_size": TILE_SIZE,
        "stride": STRIDE,
        "elapsed_seconds": round(elapsed, 3),
        "tiles_per_second": round(len(tile_rows) / elapsed, 3) if elapsed else None,
        "ground_truth_available": False,
        "accuracy_interpretation": "Cannot be calculated on this unannotated sample slide.",
    }

    (OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

    print(json.dumps(summary, indent=2))
    display(tile_df)
    display(final_df)
    plt.figure(figsize=(8, 10))
    plt.imshow(overlay)
    plt.title("All Tiles, No Tissue Filter — Unvalidated Predictions")
    plt.axis("off")
    plt.show()

    print("Saved outputs to:", OUTPUT_DIR)

finally:
    slide.close()

In [ ]:
from pathlib import Path
import json
import time

import cv2
import numpy as np
import pandas as pd
import openslide
from PIL import Image
from ultralytics import YOLO

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
SLIDE_PATH = PROJECT_DIR / "data/wsi_demo/CMU-1-Small-Region.svs"
CHECKPOINT = PROJECT_DIR / "runs/detect/baseline_yolov8n/weights/best.pt"
OUTPUT_DIR = PROJECT_DIR / "results/wsi/tissue_fraction_diagnostic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TILE_SIZE = 640
STRIDE = 512
CONFIDENCE = 0.35
TISSUE_FRACTION_THRESHOLD = 0.10

assert SLIDE_PATH.exists() and CHECKPOINT.exists()

def build_tissue_mask(thumbnail):
    rgb = np.asarray(thumbnail.convert("RGB"))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return (hsv[:, :, 1] > 15) & (gray < 245)

model = YOLO(str(CHECKPOINT))
slide = openslide.OpenSlide(str(SLIDE_PATH))

tile_records = []
raw_detections = []
started = time.perf_counter()

try:
    width, height = slide.dimensions
    thumbnail = slide.get_thumbnail((2048, 2048)).convert("RGB")
    mask = build_tissue_mask(thumbnail)

    thumb_w, thumb_h = thumbnail.size
    scale_x = thumb_w / width
    scale_y = thumb_h / height

    for y in range(0, height, STRIDE):
        for x in range(0, width, STRIDE):
            tile_w = min(TILE_SIZE, width - x)
            tile_h = min(TILE_SIZE, height - y)

            if tile_w <= 0 or tile_h <= 0:
                continue

            # Approximate the tile footprint in thumbnail coordinates.
            tx1 = max(0, min(thumb_w, int(x * scale_x)))
            ty1 = max(0, min(thumb_h, int(y * scale_y)))
            tx2 = max(tx1 + 1, min(thumb_w, int((x + tile_w) * scale_x)))
            ty2 = max(ty1 + 1, min(thumb_h, int((y + tile_h) * scale_y)))

            mask_region = mask[ty1:ty2, tx1:tx2]
            tissue_fraction = (
                float(mask_region.mean()) if mask_region.size else 0.0
            )

            process_tile = tissue_fraction >= TISSUE_FRACTION_THRESHOLD

            tile_records.append({
                "tile_x": x,
                "tile_y": y,
                "tile_width": tile_w,
                "tile_height": tile_h,
                "tissue_fraction": tissue_fraction,
                "process": process_tile,
                "detections": 0,
            })

            if not process_tile:
                continue

            region = slide.read_region(
                (x, y), 0, (tile_w, tile_h)
            ).convert("RGB")

            padded = Image.new("RGB", (TILE_SIZE, TILE_SIZE), (255, 255, 255))
            padded.paste(region, (0, 0))

            result = model.predict(
                source=np.asarray(padded),
                imgsz=TILE_SIZE,
                conf=CONFIDENCE,
                iou=0.7,
                max_det=300,
                device=0,
                verbose=False,
            )[0]

            if result.boxes is None or len(result.boxes) == 0:
                continue

            boxes = result.boxes.xyxy.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)

            kept = 0
            for box, score, cls in zip(boxes, scores, classes):
                x1, y1, x2, y2 = map(float, box)
                x1, x2 = np.clip([x1, x2], 0, tile_w)
                y1, y2 = np.clip([y1, y2], 0, tile_h)

                if x2 <= x1 or y2 <= y1:
                    continue

                raw_detections.append([
                    x + x1, y + y1, x + x2, y + y2,
                    float(score), int(cls), x, y
                ])
                kept += 1

            tile_records[-1]["detections"] = kept

    tiles_df = pd.DataFrame(tile_records)
    tiles_df.to_csv(OUTPUT_DIR / "tile_filter_results.csv", index=False)

    raw_df = pd.DataFrame(
        raw_detections,
        columns=[
            "x1", "y1", "x2", "y2", "confidence",
            "class_id", "tile_x", "tile_y"
        ],
    )
    raw_df.to_csv(OUTPUT_DIR / "raw_detections.csv", index=False)

    # Reuse the tested global NMS utility.
    import sys
    sys.path.insert(0, str(PROJECT_DIR / "src"))
    from wsi_inference import global_nms

    nms_input = raw_df[
        ["x1", "y1", "x2", "y2", "confidence", "class_id"]
    ].values.tolist()

    final_detections = global_nms(nms_input, iou_threshold=0.5)

    pd.DataFrame(
        final_detections,
        columns=["x1", "y1", "x2", "y2", "confidence", "class_id"],
    ).to_csv(OUTPUT_DIR / "final_detections.csv", index=False)

    elapsed = time.perf_counter() - started

    summary = {
        "total_tiles": int(len(tiles_df)),
        "tiles_processed": int(tiles_df["process"].sum()),
        "tiles_skipped": int((~tiles_df["process"]).sum()),
        "tissue_fraction_threshold": TISSUE_FRACTION_THRESHOLD,
        "detections_before_global_nms": int(len(raw_df)),
        "detections_after_global_nms": int(len(final_detections)),
        "elapsed_seconds": round(elapsed, 3),
        "tiles_per_second": round(len(tiles_df) / elapsed, 3) if elapsed else None,
        "ground_truth_available": False,
    }

    (OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    display(tiles_df)

finally:
    slide.close()

In [ ]:
from pathlib import Path
import json
import yaml

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")

# Check likely checkpoint locations without assuming one exists.
checkpoint_candidates = [
    PROJECT_DIR / "checkpoints/best.pt",
    PROJECT_DIR / "runs/detect/baseline_yolov8n/weights/best.pt",
    PROJECT_DIR / "runs/baseline_yolov8n/weights/best.pt",
]

print("CHECKPOINT SEARCH")
existing_checkpoints = []

for path in checkpoint_candidates:
    exists = path.exists()
    print(f"{'FOUND' if exists else 'Missing'}: {path}")
    if exists:
        existing_checkpoints.append(path)

# Search project-owned folders if the expected locations differ.
for folder in [PROJECT_DIR / "checkpoints", PROJECT_DIR / "runs"]:
    if folder.exists():
        for path in folder.rglob("best.pt"):
            if path not in existing_checkpoints:
                existing_checkpoints.append(path)
                print(f"FOUND (recursive search): {path}")

print("\nDATASET CONFIGURATION")
data_yaml = PROJECT_DIR / "configs/mitosis_data.yaml"
assert data_yaml.exists(), f"Missing dataset YAML: {data_yaml}"

with open(data_yaml, "r") as f:
    config = yaml.safe_load(f)

dataset_root = Path(config["path"])
print("Dataset root:", dataset_root)
print("Classes:", config.get("names"))
print("Number of classes:", config.get("nc"))

for split, relative_path in [
    ("train", config["train"]),
    ("val", config["val"]),
    ("test", config["test"]),
]:
    split_dir = dataset_root / relative_path
    images = [
        p for p in split_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    ] if split_dir.exists() else []

    print(f"{split}: {len(images)} images | exists={split_dir.exists()}")

print("\nSAVED THRESHOLD")
threshold_path = PROJECT_DIR / "results/metrics/selected_confidence_threshold.json"

if threshold_path.exists():
    print(threshold_path.read_text())
else:
    print("Threshold JSON not found; use the documented validation-selected value 0.35 only if confirmed.")

if not existing_checkpoints:
    print("\nSTOP: No checkpoint found. We need to locate or retrain the model before evaluation.")
else:
    print("\nCheckpoint candidates available:", len(existing_checkpoints))

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA_YAML = PROJECT_DIR / "configs/mitosis_data.yaml"
CHECKPOINT = (
    PROJECT_DIR / "runs/detect/baseline_yolov8n/weights/best.pt"
)
OUT_DIR = PROJECT_DIR / "results/metrics/test_evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONF = 0.35
IOU_MATCH = 0.50
PREDICTION_NMS_IOU = 0.70
IMGSZ = 640

assert CHECKPOINT.exists(), f"Checkpoint missing: {CHECKPOINT}"
assert DATA_YAML.exists(), f"Dataset YAML missing: {DATA_YAML}"

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

dataset_root = Path(data_cfg["path"])
test_dir = dataset_root / data_cfg["test"]
label_dir = dataset_root / data_cfg["test"].replace("images", "labels")

assert test_dir.exists(), f"Test images missing: {test_dir}"
assert label_dir.exists(), f"Test labels missing: {label_dir}"

test_images = sorted(
    p for p in test_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
)
assert len(test_images) > 0, "No test images found."

# Parse YOLO ground-truth labels: class x_center y_center width height,
# with coordinates normalized to [0, 1].
def load_gt_boxes(label_path, image_width, image_height):
    boxes = []
    if not label_path.exists():
        return boxes

    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls, xc, yc, bw, bh = map(float, parts)
        boxes.append([
            (xc - bw / 2) * image_width,
            (yc - bh / 2) * image_height,
            (xc + bw / 2) * image_width,
            (yc + bh / 2) * image_height,
            int(cls),
        ])
    return boxes

def box_iou_xyxy(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)

    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def greedy_match(gt_boxes, pred_boxes, iou_threshold=0.5):
    # Each GT and each prediction can be matched at most once.
    candidates = []
    for gi, gt in enumerate(gt_boxes):
        for pi, pred in enumerate(pred_boxes):
            if gt[4] != pred[5]:
                continue
            iou = box_iou_xyxy(gt[:4], pred[:4])
            if iou >= iou_threshold:
                candidates.append((iou, gi, pi))

    candidates.sort(reverse=True)
    used_gt, used_pred = set(), set()
    for iou, gi, pi in candidates:
        if gi not in used_gt and pi not in used_pred:
            used_gt.add(gi)
            used_pred.add(pi)

    tp = len(used_gt)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    return tp, fp, fn

model = YOLO(str(CHECKPOINT))

# Ultralytics' built-in test metrics at the fixed threshold.
print("Running Ultralytics test evaluation...")
start = time.perf_counter()
val_results = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    conf=CONF,
    iou=PREDICTION_NMS_IOU,
    device=0,
    plots=True,
    project=str(OUT_DIR / "ultralytics"),
    name="fixed_threshold_035",
    exist_ok=True,
    verbose=True,
)
ultralytics_seconds = time.perf_counter() - start

ultra_metrics = {
    "precision": float(val_results.box.mp),
    "recall": float(val_results.box.mr),
    "map50": float(val_results.box.map50),
    "map50_95": float(val_results.box.map),
    "confidence_threshold": CONF,
    "prediction_nms_iou": PREDICTION_NMS_IOU,
    "imgsz": IMGSZ,
    "evaluation_seconds": round(ultralytics_seconds, 3),
}
(OUT_DIR / "ultralytics_metrics.json").write_text(
    json.dumps(ultra_metrics, indent=2)
)

# Custom predictions and one-to-one matching at the fixed confidence threshold.
per_image_rows = []
prediction_rows = []
gt_total = 0
pred_total = 0
TP = FP = FN = 0

for image_path in test_images:
    image = plt.imread(image_path)
    h, w = image.shape[:2]

    label_path = label_dir / f"{image_path.stem}.txt"
    gt_boxes = load_gt_boxes(label_path, w, h)
    gt_total += len(gt_boxes)

    pred_result = model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=CONF,
        iou=PREDICTION_NMS_IOU,
        device=0,
        verbose=False,
    )[0]

    preds = []
    if pred_result.boxes is not None and len(pred_result.boxes):
        coords = pred_result.boxes.xyxy.cpu().numpy()
        scores = pred_result.boxes.conf.cpu().numpy()
        classes = pred_result.boxes.cls.cpu().numpy().astype(int)

        for coords_i, score, cls in zip(coords, scores, classes):
            preds.append([*map(float, coords_i), float(score), int(cls)])

    tp, fp, fn = greedy_match(gt_boxes, preds, IOU_MATCH)
    TP += tp
    FP += fp
    FN += fn
    pred_total += len(preds)

    per_image_rows.append({
        "image": image_path.name,
        "ground_truth": len(gt_boxes),
        "predictions": len(preds),
        "TP": tp, "FP": fp, "FN": fn,
    })

    for p in preds:
        prediction_rows.append({
            "image": image_path.name,
            "x1": p[0], "y1": p[1], "x2": p[2], "y2": p[3],
            "confidence": p[4], "class_id": p[5],
        })

precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0.0
)

custom_metrics = {
    "test_images": len(test_images),
    "ground_truth_objects": gt_total,
    "predictions": pred_total,
    "TP": TP, "FP": FP, "FN": FN,
    "precision": precision,
    "recall": recall,
    "F1": f1,
    "confidence_threshold": CONF,
    "ground_truth_iou_threshold": IOU_MATCH,
    "prediction_nms_iou": PREDICTION_NMS_IOU,
    "matching": "Greedy one-to-one IoU matching; highest IoU pairs matched first",
    "prior_test_exposure": (
        "The test split was evaluated in earlier project work; "
        "this is not an untouched test set."
    ),
}

pd.DataFrame(per_image_rows).to_csv(
    OUT_DIR / "per_image_custom_metrics.csv", index=False
)
pd.DataFrame(prediction_rows).to_csv(
    OUT_DIR / "custom_test_predictions.csv", index=False
)
(OUT_DIR / "custom_metrics.json").write_text(
    json.dumps(custom_metrics, indent=2)
)

print("\nULTRALYTICS METRICS")
print(json.dumps(ultra_metrics, indent=2))
print("\nCUSTOM ONE-TO-ONE MATCHING METRICS")
print(json.dumps(custom_metrics, indent=2))
print("\nSaved evaluation outputs to:", OUT_DIR)

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path("/content/drive/MyDrive/mitosis_detection_project")
EVAL_DIR = PROJECT_DIR / "results/metrics/test_evaluation"
FIG_DIR = PROJECT_DIR / "results/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

per_image_path = EVAL_DIR / "per_image_custom_metrics.csv"
predictions_path = EVAL_DIR / "custom_test_predictions.csv"

assert per_image_path.exists(), f"Missing: {per_image_path}"
assert predictions_path.exists(), f"Missing: {predictions_path}"

per_image = pd.read_csv(per_image_path)
predictions = pd.read_csv(predictions_path)

per_image["total_errors"] = per_image["FP"] + per_image["FN"]
per_image["error_type"] = per_image.apply(
    lambda r: (
        "FP and FN" if r["FP"] > 0 and r["FN"] > 0
        else "FP only" if r["FP"] > 0
        else "FN only" if r["FN"] > 0
        else "No errors"
    ),
    axis=1,
)

per_image = per_image.sort_values(
    ["total_errors", "FN", "FP"],
    ascending=False
)

display(per_image.head(15))

out_csv = EVAL_DIR / "ranked_test_error_analysis.csv"
per_image.to_csv(out_csv, index=False)

# Aggregate error counts per image.
fig, ax = plt.subplots(figsize=(12, 5))
top = per_image.head(12).sort_values("total_errors")
ax.barh(top["image"], top["FN"], label="False negatives")
ax.barh(
    top["image"], top["FP"],
    left=top["FN"], label="False positives"
)
ax.set_xlabel("Number of unmatched objects")
ax.set_title("Test-set Error Analysis at Confidence 0.35")
ax.legend()
plt.tight_layout()

plot_path = FIG_DIR / "test_error_analysis.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", out_csv)
print("Saved:", plot_path)
print("\nTotal FP:", int(per_image["FP"].sum()))
print("Total FN:", int(per_image["FN"].sum()))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import yaml

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA_YAML = PROJECT / "configs/mitosis_data.yaml"
CHECKPOINT = PROJECT / "runs/detect/baseline_yolov8n/weights/best.pt"
EVAL_DIR = PROJECT / "results/metrics/test_evaluation"
OUT_DIR = PROJECT / "results/figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONF = 0.35
IMGSZ = 640
NMS_IOU = 0.7

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

root = Path(cfg["path"])
test_dir = root / cfg["test"]
label_dir = root / cfg["test"].replace("images", "labels")

ranked = pd.read_csv(EVAL_DIR / "ranked_test_error_analysis.csv")
pred_table = pd.read_csv(EVAL_DIR / "custom_test_predictions.csv")

# Prioritize missed objects and images with both error types.
selected = ranked[
    (ranked["FN"] > 0) |
    ((ranked["FP"] > 0) & (ranked["FN"] > 0))
].head(6)

model = YOLO(str(CHECKPOINT))

def read_ground_truth(label_path, width, height):
    boxes = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            vals = line.split()
            if len(vals) != 5:
                continue
            cls, xc, yc, bw, bh = map(float, vals)
            boxes.append((
                (xc - bw/2)*width,
                (yc - bh/2)*height,
                bw*width,
                bh*height,
                int(cls)
            ))
    return boxes

fig, axes = plt.subplots(len(selected), 2, figsize=(14, 4 * len(selected)))
if len(selected) == 1:
    axes = np.array([axes])

for row_idx, (_, row) in enumerate(selected.iterrows()):
    image_path = test_dir / row["image"]
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    gt = read_ground_truth(label_dir / f"{image_path.stem}.txt", width, height)

    result = model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=CONF,
        iou=NMS_IOU,
        device=0,
        verbose=False
    )[0]

    ax_gt, ax_pred = axes[row_idx]
    for ax in (ax_gt, ax_pred):
        ax.imshow(image)
        ax.axis("off")

    # Ground truth: green
    for x, y, bw, bh, cls in gt:
        ax_gt.add_patch(patches.Rectangle(
            (x, y), bw, bh, linewidth=1.8,
            edgecolor="lime", facecolor="none"
        ))
    ax_gt.set_title(f"Ground truth | {row['image']}\nGT={len(gt)}")

    # Predictions: red
    if result.boxes is not None:
        for box, score in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.conf.cpu().numpy()
        ):
            x1, y1, x2, y2 = box
            ax_pred.add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1, linewidth=1.8,
                edgecolor="red", facecolor="none"
            ))
            ax_pred.text(
                x1, max(0, y1-4), f"{score:.2f}",
                color="white", fontsize=8,
                bbox=dict(facecolor="red", alpha=0.8, pad=1)
            )

    ax_pred.set_title(
        f"Predictions | P={int(row['predictions'])}, "
        f"TP={int(row['TP'])}, FP={int(row['FP'])}, FN={int(row['FN'])}"
    )

plt.tight_layout()
output_path = OUT_DIR / "worst_test_cases_ground_truth_vs_predictions.png"
plt.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", output_path)

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA_YAML = PROJECT / "configs/mitosis_data.yaml"
CHECKPOINT = PROJECT / "runs/detect/baseline_yolov8n/weights/best.pt"
OUT_DIR = PROJECT / "results/metrics/validation_threshold_experiment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

root = Path(cfg["path"])
val_dir = root / cfg["val"]
label_dir = root / cfg["val"].replace("images", "labels")

model = YOLO(str(CHECKPOINT))
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40]

# Reuse the same custom matching protocol from the test evaluation.
# If greedy_match is not defined in this runtime, rerun its definition
# from the earlier test-evaluation cell before executing this cell.

rows = []

for conf in thresholds:
    TP = FP = FN = 0

    for image_path in sorted(val_dir.iterdir()):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        from PIL import Image
        image = Image.open(image_path).convert("RGB")
        width, height = image.size

        gt = load_gt_boxes(
            label_dir / f"{image_path.stem}.txt",
            width, height
        )

        result = model.predict(
            source=str(image_path),
            imgsz=640,
            conf=conf,
            iou=0.7,
            device=0,
            verbose=False
        )[0]

        preds = []
        if result.boxes is not None and len(result.boxes):
            coords = result.boxes.xyxy.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy().astype(int)

            for box, score, cls in zip(coords, scores, classes):
                preds.append([*map(float, box), float(score), int(cls)])

        tp, fp, fn = greedy_match(gt, preds, iou_threshold=0.5)
        TP += tp
        FP += fp
        FN += fn

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    rows.append({
        "confidence_threshold": conf,
        "TP": TP, "FP": FP, "FN": FN,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    })

results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "threshold_comparison.csv", index=False)
display(results.round(4))
print("Saved:", OUT_DIR / "threshold_comparison.csv")

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch
import yaml

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")

CHECKPOINT = (
    PROJECT / "runs/detect/baseline_yolov8n/weights/best.pt"
)
RUN_DIR = PROJECT / "runs/detect/baseline_yolov8n"
RESULTS_CSV = RUN_DIR / "results.csv"
CONFIG_JSON = PROJECT / "configs/baseline_training_config.json"
DATA_YAML = PROJECT / "configs/mitosis_data.yaml"

print("=" * 65)
print("1. TRAINING CONFIGURATION")
print("=" * 65)

if CONFIG_JSON.exists():
    config = json.loads(CONFIG_JSON.read_text())
    print(json.dumps(config, indent=2, default=str))
else:
    print("Training config JSON not found:", CONFIG_JSON)

print("\n" + "=" * 65)
print("2. CHECKPOINT")
print("=" * 65)

if CHECKPOINT.exists():
    print("Checkpoint:", CHECKPOINT)
    print(f"Size: {CHECKPOINT.stat().st_size / (1024**2):.2f} MiB")
else:
    print("Expected checkpoint missing. Search the runs directory.")

print("\n" + "=" * 65)
print("3. TRAINING HISTORY")
print("=" * 65)

if RESULTS_CSV.exists():
    history = pd.read_csv(RESULTS_CSV)
    history.columns = history.columns.str.strip()

    print("Epochs recorded:", len(history))
    print("Columns:", history.columns.tolist())

    metric_cols = [
        c for c in history.columns
        if any(term in c.lower() for term in [
            "epoch", "precision", "recall", "map50",
            "map95", "box_loss", "cls_loss", "dfl_loss"
        ])
    ]

    display(history[metric_cols].tail(8).round(4))

    map_cols = [
        c for c in history.columns
        if "map50-95" in c.lower() or "map50_95" in c.lower()
    ]
    if map_cols:
        best_idx = history[map_cols[0]].astype(float).idxmax()
        print("\nBest recorded mAP@0.50:0.95 epoch:")
        print(history.loc[best_idx, metric_cols].to_dict())
else:
    print("Training history CSV not found:", RESULTS_CSV)

print("\n" + "=" * 65)
print("4. AVAILABLE RUN ARTIFACTS")
print("=" * 65)

for folder in [RUN_DIR, RUN_DIR / "weights"]:
    if folder.exists():
        print(f"\n{folder}")
        for p in sorted(folder.iterdir()):
            if p.is_file():
                print(f"  {p.name} — {p.stat().st_size / (1024**2):.2f} MiB")

print("\n" + "=" * 65)
print("5. GPU")
print("=" * 65)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / (1024**3), 2),
        "GiB"
    )

print("\nAudit complete. No training or test evaluation was performed.")

In [ ]:
from pathlib import Path
import yaml

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
RUN_DIR = PROJECT / "runs/detect/baseline_yolov8n"
ARGS_PATH = RUN_DIR / "args.yaml"

assert ARGS_PATH.exists(), f"Cannot find {ARGS_PATH}"

with open(ARGS_PATH) as f:
    args = yaml.safe_load(f)

keys_to_inspect = [
    "model", "data", "imgsz", "epochs", "batch",
    "patience", "seed", "deterministic", "workers",
    "optimizer", "lr0", "lrf", "weight_decay",
    "warmup_epochs", "close_mosaic", "mosaic",
    "mixup", "copy_paste", "degrees", "translate",
    "scale", "fliplr", "flipud", "hsv_h", "hsv_s", "hsv_v"
]

for key in keys_to_inspect:
    print(f"{key}: {args.get(key, '(not recorded)')}")

print("\nBaseline checkpoint:")
print(RUN_DIR / "weights/best.pt")
print("Exists:", (RUN_DIR / "weights/best.pt").exists())

In [ ]:
from pathlib import Path
import json
import time
import yaml
import torch
from ultralytics import YOLO

# ---------- Persistent project paths ----------
PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA_YAML = PROJECT / "configs/mitosis_data.yaml"
RUNS_DIR = PROJECT / "runs"
CONFIG_DIR = PROJECT / "configs"
LOG_DIR = PROJECT / "logs"

RUN_NAME = "augmentation_experiment_v1"
RUN_DIR = RUNS_DIR / "detect" / RUN_NAME
CHECKPOINT_DIR = PROJECT / "checkpoints"

BASELINE_BEST = (
    PROJECT / "runs/detect/baseline_yolov8n/weights/best.pt"
)

assert DATA_YAML.exists(), f"Dataset YAML missing: {DATA_YAML}"
assert BASELINE_BEST.exists(), "Baseline checkpoint missing."
assert torch.cuda.is_available(), "GPU unavailable. Enable a GPU runtime first."
assert not RUN_DIR.exists(), (
    f"{RUN_DIR} already exists. Choose a new run name to avoid overwriting."
)

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

# Confirm the dataset paths still resolve.
dataset_root = Path(data_cfg["path"])
for split in [data_cfg["train"], data_cfg["val"]]:
    assert (dataset_root / split).exists(), (
        f"Missing dataset split: {dataset_root / split}"
    )

# ---------- Experiment configuration ----------
experiment = {
    "experiment": RUN_NAME,
    "hypothesis": (
        "Conservative augmentation may improve detection robustness "
        "by reducing aggressive scale, mosaic, and color variation."
    ),
    "initial_weights": "yolov8n.pt",
    "imgsz": 640,
    "epochs": 30,
    "batch": 8,
    "patience": 8,
    "seed": 42,
    "deterministic": True,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "close_mosaic": 8,
    "mosaic": 0.5,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "degrees": 0.0,
    "translate": 0.05,
    "scale": 0.25,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.4,
    "hsv_v": 0.25,
    "data": str(DATA_YAML),
    "baseline_checkpoint_for_comparison": str(BASELINE_BEST),
    "evaluation_policy": (
        "Select model using validation data only. Keep confidence threshold "
        "selection on validation. Do not use test metrics to choose the model."
    ),
}

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
config_path = CONFIG_DIR / f"{RUN_NAME}.json"
config_path.write_text(json.dumps(experiment, indent=2))

print("Experiment:", RUN_NAME)
print("GPU:", torch.cuda.get_device_name(0))
print("Configuration saved:", config_path)

# ---------- Train ----------
# Start from the same pretrained initialization as the baseline.
model = YOLO("yolov8n.pt")

start_time = time.perf_counter()

results = model.train(
    data=str(DATA_YAML),
    imgsz=experiment["imgsz"],
    epochs=experiment["epochs"],
    batch=experiment["batch"],
    patience=experiment["patience"],
    seed=experiment["seed"],
    deterministic=experiment["deterministic"],
    workers=2,
    pretrained=True,
    optimizer=experiment["optimizer"],
    lr0=experiment["lr0"],
    lrf=experiment["lrf"],
    weight_decay=experiment["weight_decay"],
    warmup_epochs=experiment["warmup_epochs"],
    close_mosaic=experiment["close_mosaic"],
    mosaic=experiment["mosaic"],
    mixup=experiment["mixup"],
    copy_paste=experiment["copy_paste"],
    degrees=experiment["degrees"],
    translate=experiment["translate"],
    scale=experiment["scale"],
    fliplr=experiment["fliplr"],
    flipud=experiment["flipud"],
    hsv_h=experiment["hsv_h"],
    hsv_s=experiment["hsv_s"],
    hsv_v=experiment["hsv_v"],
    save=True,
    save_period=5,
    plots=True,
    project=str(RUNS_DIR / "detect"),
    name=RUN_NAME,
    exist_ok=False,
    device=0,
    verbose=True,
)

elapsed = time.perf_counter() - start_time

# ---------- Verify saved artifacts ----------
best_path = RUN_DIR / "weights" / "best.pt"
last_path = RUN_DIR / "weights" / "last.pt"
results_csv = RUN_DIR / "results.csv"

summary = {
    "experiment": RUN_NAME,
    "elapsed_seconds": round(elapsed, 2),
    "best_checkpoint_exists": best_path.exists(),
    "last_checkpoint_exists": last_path.exists(),
    "training_history_exists": results_csv.exists(),
    "best_checkpoint": str(best_path),
    "last_checkpoint": str(last_path),
    "results_csv": str(results_csv),
}

(LOG_DIR / f"{RUN_NAME}_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\nTRAINING SUMMARY")
print(json.dumps(summary, indent=2))

assert best_path.exists(), (
    "Best checkpoint not found. Inspect the run directory before proceeding."
)
assert results_csv.exists(), "Training history was not saved."

# Extra copy of best checkpoint into the dedicated checkpoint directory.
# Keep the original run artifact intact.
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
backup_path = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"

import shutil
shutil.copy2(best_path, backup_path)
print("Backup checkpoint:", backup_path)

In [ ]:
from pathlib import Path
import pandas as pd
import json
from ultralytics import YOLO

PROJECT = Path("/content/drive/MyDrive/mitosis_detection_project")
DATA = str(PROJECT / "configs/mitosis_data.yaml")

models = {
    "baseline": PROJECT / "runs/detect/baseline_yolov8n/weights/best.pt",
    "augmentation_v1": PROJECT / "runs/detect/augmentation_experiment_v1/weights/best.pt",
}

comparison = []

for name, checkpoint in models.items():
    assert checkpoint.exists(), f"Missing checkpoint: {checkpoint}"

    model = YOLO(str(checkpoint))
    metrics = model.val(
        data=DATA,
        split="val",
        imgsz=640,
        batch=8,
        conf=0.001,
        iou=0.7,
        plots=False,
        verbose=False,
        project=str(PROJECT / "results/metrics"),
        name=f"final_validation_{name}",
        exist_ok=True,
    )

    comparison.append({
        "model": name,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    })

df = pd.DataFrame(comparison)
out = PROJECT / "results/metrics/final_model_comparison.csv"
df.to_csv(out, index=False)

display(df)
print("Saved:", out)